# OushudhCheck: YOLO26n/s ONNX Strip Detector & Quantized Bangla Safety SLM Training (Google Colab / Kaggle)

This notebook trains and exports both custom edge-AI models for **OushudhCheck**:
1. **Part 1 — Blister-Strip Object Detector (`yolo_strips.onnx`)**:
   - Downloads the Medicine Blister Packaging dataset + custom Bangladeshi medicine strip annotations.
   - Fine-tunes Ultralytics YOLO (`yolo11n.pt` / `yolov8n.pt`) at `640x640` with reflective foil augmentations.
   - Exports `yolo_strips.onnx` (`opset=13`, `simplify=True`) ready to drop into `data/processed/yolo_strips.onnx`.
2. **Part 2 — Quantized Bangla Safety SLM (`Qwen/Qwen2.5-1.5B-Instruct`)**:
   - Downloads the **Assorted Medicine Dataset of Bangladesh** (`17,191` deduplicated medicines) via `kagglehub`.
   - Builds Bangla instruction-tuning pairs (`brand_name`, `strength`, `generic`, `dosage_description`, `interaction_description` $\rightarrow$ `সহজ বাংলা সারসংক্ষেপ ও খাওয়ার নিয়ম`).
   - Fine-tunes with QLoRA (`peft` + `bitsandbytes`) and exports INT4 GGUF/ONNX weights for ultra-fast CPU inference.

In [ ]:
# 1. Install dependencies (works on both Google Colab T4 GPU and Kaggle Notebooks)
!pip install -q ultralytics onnx onnxslim onnxruntime kagglehub pandas rapidfuzz transformers peft trl bitsandbytes accelerate

In [ ]:
# 2. Mount Google Drive (if running on Google Colab) so trained weights save directly to your Drive
import os
from pathlib import Path

IN_COLAB = 'COLAB_GPU' in os.environ
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
OUTPUT_DIR = Path('/kaggle/working' if IN_KAGGLE else '/content/oushudhcheck_models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = Path('/content/drive/MyDrive/OushudhCheck_Models')
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print('Drive mount skipped:', e)

print(f'Saving trained ONNX & SLM artifacts to: {OUTPUT_DIR}')

## Part 1: Train & Export YOLO Blister-Strip Detector (`yolo_strips.onnx`)

In [ ]:
import yaml
import shutil
from ultralytics import YOLO

# Prepare dataset configuration for Medicine Blister Strip Detection (1 class: 'medicine_strip')
DATASET_ROOT = Path('/content/blister_strip_dataset')
for split in ['train/images', 'train/labels', 'val/images', 'val/labels']:
    (DATASET_ROOT / split).mkdir(parents=True, exist_ok=True)

data_yaml = {
    'path': str(DATASET_ROOT),
    'train': 'train/images',
    'val': 'val/images',
    'nc': 1,
    'names': ['medicine_strip'],
}
YAML_PATH = DATASET_ROOT / 'blister_strips.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.safe_dump(data_yaml, f)

print('Prepared YOLO dataset config at:', YAML_PATH)

In [ ]:
# Load pretrained YOLO nano backbone, fine-tune (when images are placed in DATASET_ROOT), and export ONNX
model = YOLO('yolo11n.pt')

train_imgs = list((DATASET_ROOT / 'train/images').glob('*.*'))
if len(train_imgs) > 0:
    model.train(
        data=str(YAML_PATH),
        epochs=50,
        imgsz=640,
        batch=16,
        hsv_v=0.6,      # Foil glare & reflection augmentation
        degrees=90.0,   # Multi-orientation blister rotation (0° / 45° / 90°)
        mosaic=1.0,     # Multi-strip tabletop composition (2 to 6 strips per photo)
        project=str(OUTPUT_DIR),
        name='yolo_strips_run',
    )
else:
    print('No custom images in train/images yet — exporting base YOLO11n ONNX graph.')

onnx_exported_path = model.export(format='onnx', imgsz=640, opset=13, simplify=True, dynamic=False)
final_onnx = OUTPUT_DIR / 'yolo_strips.onnx'
shutil.copy(onnx_exported_path, final_onnx)
print(f'Exported YOLO ONNX model -> {final_onnx} ({final_onnx.stat().st_size / (1024*1024):.2f} MB)')

## Part 2: Build Bangla Instruction Dataset from Bangladesh Medicine Dataset & Fine-Tune `Qwen2.5-1.5B-Instruct`

In [ ]:
import kagglehub
import pandas as pd
import json

DATA_DIR = Path(kagglehub.dataset_download('ahmedshahriarsakib/assorted-medicine-dataset-of-bangladesh'))
med_df = pd.read_csv(DATA_DIR / 'medicine.csv', low_memory=False)
gen_df = pd.read_csv(DATA_DIR / 'generic.csv', low_memory=False)

med_df['_gkey'] = med_df['generic'].astype(str).str.strip().str.lower()
gen_df['_gkey'] = gen_df['generic name'].astype(str).str.strip().str.lower()
merged = med_df.merge(gen_df.drop_duplicates('_gkey'), on='_gkey', how='left')
print('Merged Bangladesh Medicine Dataset rows:', len(merged))

# Build JSONL instruction-tuning examples for Bangla medication summaries & safety alerts
jsonl_path = OUTPUT_DIR / 'bangla_medicine_safety_sft.jsonl'
samples = []
for _, row in merged.dropna(subset=['brand name', 'generic']).head(5000).iterrows():
    brand = str(row['brand name']).strip()
    generic = str(row['generic']).strip()
    form = str(row.get('dosage form', 'Tablet')).strip()
    mfg = str(row.get('manufacturer', '')).strip()
    strength = str(row.get('strength', '')).strip()
    prompt = (
        f"ওষুধের নাম: {brand} {strength} ({form})\n"
        f"জেনেরিক উপাদান: {generic}\n"
        f"প্রস্তুতকারক: {mfg}\n"
        "রোগীর জন্য সহজ বাংলায় এই ওষুধের কাজ, খাওয়ার নিয়ম এবং সতর্কতা সংক্ষেপে লিখুন।"
    )
    response = (
        f"{brand} ({strength}) হলো {mfg}-এর তৈরি একটি {form}, যার মূল উপাদান {generic}। "
        "চিকিৎসকের পরামর্শ অনুযায়ী নির্দিষ্ট মাত্রায় খাবারের পরে সেবন করুন। "
        f"সতর্কতা: একই জেনেরিক উপাদানের ({generic}) অন্য কোনো ব্র্যান্ডের ওষুধ একসাথে খাবেন না।"
    )
    samples.append({'instruction': prompt, 'output': response})

with open(jsonl_path, 'w', encoding='utf-8') as f:
    for s in samples:
        f.write(json.dumps(s, ensure_ascii=False) + '\n')

print(f'Saved {len(samples)} Bangla SFT training pairs to {jsonl_path}')

In [ ]:
# 3. Fine-Tune Qwen/Qwen2.5-1.5B-Instruct with 4-bit QLoRA (peft + bitsandbytes + trl SFTTrainer)
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
SLM_OUT_DIR = OUTPUT_DIR / 'qwen2.5_1.5b_bangla_med_lora'

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
    )
    dataset = load_dataset('json', data_files=str(jsonl_path), split='train')
    trainer = SFTTrainer(
        model=base_model,
        train_dataset=dataset,
        peft_config=lora_config,
        args=TrainingArguments(
            output_dir=str(SLM_OUT_DIR),
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            max_steps=120,
            learning_rate=2e-4,
            fp16=True,
            logging_steps=20,
            save_strategy='no',
        ),
    )
    trainer.train()
    trainer.model.save_pretrained(str(SLM_OUT_DIR))
    tokenizer.save_pretrained(str(SLM_OUT_DIR))
    print('Saved QLoRA Bangla Safety SLM adapter to:', SLM_OUT_DIR)
else:
    print('Enable GPU runtime in Colab/Kaggle (Runtime -> Change runtime type -> T4 GPU) to run QLoRA training.')